Evaluates how the distance between each NHDA and its matched reference area influences the environmental comparison by assessing changes in the environmental indicators across different search radii.

In [ ]:
import math
import random
import re
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import seaborn as sns
from shapely.geometry import Point
from shapely.ops import unary_union


# ============================================================================
# CONFIGURATION
# ============================================================================

NHDA_GPKG = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\New_Housing_Development_Areas\NHDA_with_construction_years_RF.gpkg"
ATKIS_GPKG = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\ATKIS\ATKIS_41001_41006.gpkg"
NEW_BLDG_GPKG = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_new_buildings.gpkg"
EXIST_BLDG_GPKG = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_existing.gpkg"

LST_BASE_PATH = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\DatasetSpecific\Landsat_8_9")
LST_YEAR = 2025
NDVI_DIR = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\DatasetSpecific\Sentinel_2_WASP_v2\NDVI_Bavaria\Bayern_Final\masked")
NDVI_PATTERN = "Bayern_NDVI_*median_25832.tif"
NDVI_YEAR = 2024

OUTPUT_DIR = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Reference_Areas\Sensitivity_Distance")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ATKIS_CLASSES = [41001, 41006]
ATKIS_CODE_COL = "OBJART"
BUFFER_DISTANCES = [100, 200, 300, 400, 500, 1000, 1500, 2000]
AREA_TOLERANCE = 0.2
GRID_SPACING = 75
NEW_VS_EXISTING_MAX = 0.1
N_RANDOM_POINTS = 100
MIN_VALID_POINT_SHARE = 0.3
N_TEST_CLUSTERS = 120  # set None to run all NHDA
RANDOM_SEED = 42
DEBUG = False

sns.set_theme(style="whitegrid", context="talk")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("=" * 90)
print("SENSITIVITY ANALYSIS: DISTANCE OF NHDA AND REFERENCE AREA")
print("=" * 90)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"LST year: {LST_YEAR}")
print(f"NDVI year: {NDVI_YEAR}")
print(f"Output: {OUTPUT_DIR}")


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================


def chunked_unary_union(geometries, chunk_size=500):
    geom_list = [geom for geom in geometries if geom is not None and not geom.is_empty]
    if not geom_list:
        return None
    if len(geom_list) <= chunk_size:
        return unary_union(geom_list)

    chunks = []
    for i in range(0, len(geom_list), chunk_size):
        chunks.append(unary_union(geom_list[i:i + chunk_size]))
    return unary_union(chunks)



def create_grid_points(polygon, spacing=100):
    minx, miny, maxx, maxy = polygon.bounds
    points = []
    for x in np.arange(minx, maxx, spacing):
        for y in np.arange(miny, maxy, spacing):
            point = Point(x, y)
            if polygon.contains(point):
                points.append(point)
    return points



def create_circle_from_area(center_point, target_area):
    radius = math.sqrt(target_area / np.pi)
    return center_point.buffer(radius)



def generate_random_points_in_geometry(geometry, n_points, max_attempts=None):
    if max_attempts is None:
        max_attempts = n_points * 100

    points = []
    attempts = 0
    minx, miny, maxx, maxy = geometry.bounds

    while len(points) < n_points and attempts < max_attempts:
        point = Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
        if geometry.contains(point):
            points.append(point)
        attempts += 1

    return points



def calculate_new_vs_existing_ratio(
    geometry,
    new_gdf,
    new_sindex,
    existing_gdf,
    existing_sindex,
    debug=False,
):
    try:
        def clipped_area(gdf, sindex):
            hits = list(sindex.intersection(geometry.bounds))
            if not hits:
                return 0.0
            candidates = gdf.iloc[hits]
            clipped = candidates.geometry.intersection(geometry)
            return float(clipped[~clipped.is_empty].area.sum())

        new_area = clipped_area(new_gdf, new_sindex)
        existing_area = clipped_area(existing_gdf, existing_sindex)

        if new_area == 0.0:
            return 0.0
        if existing_area == 0.0:
            return float("inf")
        return new_area / existing_area

    except Exception as exc:
        if debug:
            print(f"      ratio error: {exc}")
        return float("inf")



def find_valid_circles(
    grid_points,
    search_polygon,
    target_area,
    new_gdf,
    new_sindex,
    existing_gdf,
    existing_sindex,
    tolerance=0.2,
    ratio_max=0.1,
):
    valid_circles = []

    for point in grid_points:
        circle = create_circle_from_area(point, target_area)
        clipped = circle.intersection(search_polygon)

        if clipped.is_empty:
            continue

        area_diff = abs(clipped.area - target_area) / target_area
        if area_diff > tolerance:
            continue

        ratio = calculate_new_vs_existing_ratio(
            clipped,
            new_gdf,
            new_sindex,
            existing_gdf,
            existing_sindex,
            debug=DEBUG,
        )
        if ratio > ratio_max:
            continue

        valid_circles.append(
            {
                "geometry": clipped,
                "center": point,
                "area": clipped.area,
                "area_diff_ratio": area_diff,
                "new_vs_existing": ratio,
            }
        )

    return valid_circles



def choose_distance_band_candidate(circles, nhda_centroid, band_min, band_max):
    if not circles:
        return None

    band_mid = (band_min + band_max) / 2.0
    candidates = []
    for circle in circles:
        center_distance = nhda_centroid.distance(circle["center"])
        if center_distance < band_min or center_distance > band_max:
            continue
        candidate = circle.copy()
        candidate["center_distance_m"] = center_distance
        candidates.append(candidate)

    if not candidates:
        return None

    candidates.sort(
        key=lambda c: (
            abs(c["center_distance_m"] - band_mid),
            c["area_diff_ratio"],
            c["new_vs_existing"],
        )
    )
    return candidates[0]



def extract_raster_values(points_gdf, raster_path, scale_divisor=None, valid_range=None):
    try:
        with rasterio.open(raster_path) as src:
            points_reproj = points_gdf.to_crs(src.crs) if points_gdf.crs != src.crs else points_gdf
            values = []
            for _, point in points_reproj.iterrows():
                coords = [(point.geometry.x, point.geometry.y)]
                for val in src.sample(coords):
                    raw = float(val[0])
                    if src.nodata is not None and raw == src.nodata:
                        values.append(np.nan)
                        continue
                    if valid_range is not None:
                        low, high = valid_range
                        if raw < low or raw > high:
                            values.append(np.nan)
                            continue
                    if scale_divisor is not None:
                        raw = raw / scale_divisor
                    values.append(raw)
            return values
    except Exception:
        return [np.nan] * len(points_gdf)



def get_lst_files(base_path, year):
    year_path_cd10 = base_path / f"{year}_cd10"
    year_path_plain = base_path / f"{year}"
    if year_path_cd10.exists():
        year_path = year_path_cd10
    elif year_path_plain.exists():
        year_path = year_path_plain
    else:
        return []
    return sorted(year_path.glob("LST_P*.tif"))



def get_ndvi_file(ndvi_dir, pattern, year):
    for file_path in sorted(ndvi_dir.glob(pattern)):
        match = re.search(r"Bayern_NDVI_(\d{4})", file_path.stem)
        if match and int(match.group(1)) == year:
            return file_path
    return None


# ============================================================================
# 1. LOAD DATA
# ============================================================================

print("\n1. LOAD DATA")
print("=" * 90)

nhda_gdf = gpd.read_file(NHDA_GPKG)
atkis_gdf = gpd.read_file(ATKIS_GPKG)
new_gdf = gpd.read_file(NEW_BLDG_GPKG)
exist_gdf = gpd.read_file(EXIST_BLDG_GPKG)

print(f"✓ NHDA: {len(nhda_gdf):,}")
print(f"✓ ATKIS: {len(atkis_gdf):,}")
print(f"✓ New buildings: {len(new_gdf):,}")
print(f"✓ Existing buildings: {len(exist_gdf):,}")

if ATKIS_CLASSES is not None:
    if ATKIS_CODE_COL not in atkis_gdf.columns:
        raise ValueError(
            f"Column '{ATKIS_CODE_COL}' not found in ATKIS. Available: {list(atkis_gdf.columns)}"
        )
    atkis_gdf[ATKIS_CODE_COL] = pd.to_numeric(atkis_gdf[ATKIS_CODE_COL], errors="coerce")
    atkis_gdf = atkis_gdf[atkis_gdf[ATKIS_CODE_COL].isin(ATKIS_CLASSES)].copy()
    print(f"✓ ATKIS filtered to {ATKIS_CLASSES}: {len(atkis_gdf):,}")

if nhda_gdf.crs is None or nhda_gdf.crs.is_geographic:
    raise ValueError("NHDA layer must use a projected CRS.")

target_crs = nhda_gdf.crs
for name, gdf in [("ATKIS", atkis_gdf), ("new", new_gdf), ("existing", exist_gdf)]:
    if gdf.crs != target_crs:
        print(f"  Reprojecting {name} to {target_crs}")

atkis_gdf = atkis_gdf.to_crs(target_crs) if atkis_gdf.crs != target_crs else atkis_gdf
new_gdf = new_gdf.to_crs(target_crs) if new_gdf.crs != target_crs else new_gdf
exist_gdf = exist_gdf.to_crs(target_crs) if exist_gdf.crs != target_crs else exist_gdf

if N_TEST_CLUSTERS is not None:
    nhda_gdf = nhda_gdf.sample(min(N_TEST_CLUSTERS, len(nhda_gdf)), random_state=RANDOM_SEED).copy()
    print(f"✓ Sampled NHDA for sensitivity test: {len(nhda_gdf):,}")
else:
    print(f"✓ Using all NHDA: {len(nhda_gdf):,}")

new_sindex = new_gdf.sindex
exist_sindex = exist_gdf.sindex

print("Building ATKIS union...")
atkis_union = chunked_unary_union(atkis_gdf.geometry, chunk_size=500)
if atkis_union is None:
    raise ValueError("ATKIS union could not be created.")
print("✓ ATKIS union ready")


# ============================================================================
# 2. DISCOVER RASTER INPUTS
# ============================================================================

print("\n2. DISCOVER RASTER INPUTS")
print("=" * 90)

lst_files = get_lst_files(LST_BASE_PATH, LST_YEAR)
ndvi_file = get_ndvi_file(NDVI_DIR, NDVI_PATTERN, NDVI_YEAR)

if not lst_files:
    raise FileNotFoundError(f"No LST rasters found for year {LST_YEAR} under {LST_BASE_PATH}")
if ndvi_file is None:
    raise FileNotFoundError(f"No NDVI raster found for year {NDVI_YEAR} under {NDVI_DIR}")

print(f"✓ LST {LST_YEAR}: {len(lst_files)} raster(s)")
print(f"✓ NDVI {NDVI_YEAR}: {ndvi_file.name}")


# ============================================================================
# 3. GENERATE DISTANCE-BANDED CANDIDATE REFERENCE AREAS
# ============================================================================

print("\n3. GENERATE DISTANCE-BANDED CANDIDATE REFERENCE AREAS")
print("=" * 90)

candidate_entries = []
failed_nhda = []

for idx, (_, row) in enumerate(nhda_gdf.iterrows(), start=1):
    nhda_id = row["nhda_id"]
    nhda_geom = row.geometry
    nhda_area = nhda_geom.area
    nhda_centroid = nhda_geom.centroid

    if idx % 20 == 0 or idx == 1:
        print(f"  Processing NHDA {idx}/{len(nhda_gdf)}")

    nhda_points = generate_random_points_in_geometry(nhda_geom, N_RANDOM_POINTS)
    if len(nhda_points) < N_RANDOM_POINTS * 0.5:
        failed_nhda.append(nhda_id)
        continue

    nhda_points_gdf = gpd.GeoDataFrame(geometry=nhda_points, crs=target_crs)
    found_any = False
    previous_buffer = 0

    for current_buffer in BUFFER_DISTANCES:
        outer_buffer = nhda_geom.buffer(current_buffer)
        inner_buffer = nhda_geom.buffer(previous_buffer)
        ring = outer_buffer.difference(inner_buffer).intersection(atkis_union).difference(nhda_geom)

        band_min = previous_buffer
        band_max = current_buffer
        previous_buffer = current_buffer

        if ring.is_empty:
            continue

        grid_points = create_grid_points(ring, spacing=GRID_SPACING)
        if not grid_points:
            continue

        circles = find_valid_circles(
            grid_points,
            ring,
            nhda_area,
            new_gdf,
            new_sindex,
            exist_gdf,
            exist_sindex,
            tolerance=AREA_TOLERANCE,
            ratio_max=NEW_VS_EXISTING_MAX,
        )
        if not circles:
            continue

        best = choose_distance_band_candidate(circles, nhda_centroid, band_min, band_max)
        if best is None:
            continue

        ra_points = generate_random_points_in_geometry(best["geometry"], N_RANDOM_POINTS)
        if len(ra_points) < N_RANDOM_POINTS * 0.5:
            continue

        candidate_entries.append(
            {
                "nhda_id": nhda_id,
                "buffer_m": current_buffer,
                "band_min_m": band_min,
                "band_max_m": band_max,
                "center_distance_m": best["center_distance_m"],
                "nhda_area_m2": nhda_area,
                "ra_area_m2": best["geometry"].area,
                "area_diff_pct": best["area_diff_ratio"] * 100,
                "new_vs_existing": best["new_vs_existing"],
                "geometry": best["geometry"],
                "nhda_points_gdf": nhda_points_gdf,
                "ra_points_gdf": gpd.GeoDataFrame(geometry=ra_points, crs=target_crs),
            }
        )
        found_any = True

    if not found_any:
        failed_nhda.append(nhda_id)

if not candidate_entries:
    raise ValueError("No valid distance-banded candidate reference areas were found.")

print(f"✓ Candidate entries: {len(candidate_entries):,}")
print(f"✓ NHDA with at least one candidate band: {pd.Series([c['nhda_id'] for c in candidate_entries]).nunique():,}")
print(f"✓ Failed NHDA: {len(set(failed_nhda)):,}")

candidate_meta = pd.DataFrame(
    [
        {
            key: value
            for key, value in entry.items()
            if key not in {"geometry", "nhda_points_gdf", "ra_points_gdf"}
        }
        for entry in candidate_entries
    ]
)

candidate_gdf = gpd.GeoDataFrame(
    candidate_meta.copy(),
    geometry=[entry["geometry"] for entry in candidate_entries],
    crs=target_crs,
)

candidate_gpkg = OUTPUT_DIR / "candidate_reference_areas_by_distance.gpkg"
candidate_gdf.to_file(candidate_gpkg, driver="GPKG")
print(f"✓ Candidate geometries: {candidate_gpkg}")


# ============================================================================
# 4. EXTRACT SINGLE-YEAR LST AND NDVI DIFFERENCES
# ============================================================================

print("\n4. EXTRACT SINGLE-YEAR DIFFERENCES")
print("=" * 90)

results = []

for entry in candidate_entries:
    nhda_lst_values_all = []
    ra_lst_values_all = []

    for raster_file in lst_files:
        nhda_lst_values_all.extend(extract_raster_values(entry["nhda_points_gdf"], raster_file))
        ra_lst_values_all.extend(extract_raster_values(entry["ra_points_gdf"], raster_file))

    nhda_lst_values = np.array(nhda_lst_values_all, dtype=float)
    ra_lst_values = np.array(ra_lst_values_all, dtype=float)
    nhda_ndvi_values = np.array(
        extract_raster_values(
            entry["nhda_points_gdf"],
            ndvi_file,
            scale_divisor=100.0,
            valid_range=(-100.0, 100.0),
        ),
        dtype=float,
    )
    ra_ndvi_values = np.array(
        extract_raster_values(
            entry["ra_points_gdf"],
            ndvi_file,
            scale_divisor=100.0,
            valid_range=(-100.0, 100.0),
        ),
        dtype=float,
    )

    n_valid_lst_nhda = int(np.sum(~np.isnan(nhda_lst_values)))
    n_valid_lst_ra = int(np.sum(~np.isnan(ra_lst_values)))
    n_valid_ndvi_nhda = int(np.sum(~np.isnan(nhda_ndvi_values)))
    n_valid_ndvi_ra = int(np.sum(~np.isnan(ra_ndvi_values)))

    if n_valid_lst_nhda < N_RANDOM_POINTS * MIN_VALID_POINT_SHARE:
        continue
    if n_valid_lst_ra < N_RANDOM_POINTS * MIN_VALID_POINT_SHARE:
        continue
    if n_valid_ndvi_nhda < N_RANDOM_POINTS * MIN_VALID_POINT_SHARE:
        continue
    if n_valid_ndvi_ra < N_RANDOM_POINTS * MIN_VALID_POINT_SHARE:
        continue

    nhda_mean_lst = float(np.nanmean(nhda_lst_values))
    ra_mean_lst = float(np.nanmean(ra_lst_values))
    diff_lst = nhda_mean_lst - ra_mean_lst

    nhda_median_ndvi = float(np.nanmedian(nhda_ndvi_values))
    ra_median_ndvi = float(np.nanmedian(ra_ndvi_values))
    diff_ndvi = nhda_median_ndvi - ra_median_ndvi

    results.append(
        {
            "nhda_id": entry["nhda_id"],
            "buffer_m": entry["buffer_m"],
            "band_min_m": entry["band_min_m"],
            "band_max_m": entry["band_max_m"],
            "center_distance_m": entry["center_distance_m"],
            "nhda_area_m2": entry["nhda_area_m2"],
            "ra_area_m2": entry["ra_area_m2"],
            "area_diff_pct": entry["area_diff_pct"],
            "new_vs_existing": entry["new_vs_existing"],
            "nhda_mean_lst_2025": nhda_mean_lst,
            "ra_mean_lst_2025": ra_mean_lst,
            "diff_lst_2025": diff_lst,
            "abs_diff_lst_2025": abs(diff_lst),
            "nhda_median_ndvi_2024": nhda_median_ndvi,
            "ra_median_ndvi_2024": ra_median_ndvi,
            "diff_ndvi_2024": diff_ndvi,
            "abs_diff_ndvi_2024": abs(diff_ndvi),
            "n_valid_lst_nhda": n_valid_lst_nhda,
            "n_valid_lst_ra": n_valid_lst_ra,
            "n_valid_ndvi_nhda": n_valid_ndvi_nhda,
            "n_valid_ndvi_ra": n_valid_ndvi_ra,
        }
    )

if not results:
    raise ValueError("No valid sensitivity results were created.")

df = pd.DataFrame(results)
print(f"✓ Valid NHDA-RA candidate pairs with both variables: {len(df):,}")


# ============================================================================
# 5. SUMMARIES AND DISTANCE RELATIONSHIPS
# ============================================================================

print("\n5. BUILD SUMMARIES")
print("=" * 90)

summary_by_buffer = (
    df.groupby("buffer_m", as_index=False)
    .agg(
        mean_abs_diff_lst_2025=("abs_diff_lst_2025", "mean"),
        median_abs_diff_lst_2025=("abs_diff_lst_2025", "median"),
        mean_abs_diff_ndvi_2024=("abs_diff_ndvi_2024", "mean"),
        median_abs_diff_ndvi_2024=("abs_diff_ndvi_2024", "median"),
        n_pairs=("nhda_id", "count"),
        n_nhda=("nhda_id", "nunique"),
    )
    .sort_values("buffer_m")
)

trend_df = pd.DataFrame(
    [
        {
            "variable": "LST_2025",
            "spearman_rho_distance_vs_absdiff": df["center_distance_m"].corr(df["abs_diff_lst_2025"], method="spearman"),
            "pearson_r_distance_vs_absdiff": df["center_distance_m"].corr(df["abs_diff_lst_2025"], method="pearson"),
            "n_rows": len(df),
        },
        {
            "variable": "NDVI_2024",
            "spearman_rho_distance_vs_absdiff": df["center_distance_m"].corr(df["abs_diff_ndvi_2024"], method="spearman"),
            "pearson_r_distance_vs_absdiff": df["center_distance_m"].corr(df["abs_diff_ndvi_2024"], method="pearson"),
            "n_rows": len(df),
        },
    ]
)

print("Summary by distance band:")
print(summary_by_buffer.to_string(index=False))
print()
print("Distance correlations:")
print(trend_df.to_string(index=False))


# ============================================================================
# 6. SAVE TABLES
# ============================================================================

print("\n6. SAVE TABLES")
print("=" * 90)

paths_to_save = {
    "candidate_metadata": OUTPUT_DIR / "candidate_reference_areas_metadata.csv",
    "pair_results": OUTPUT_DIR / "distance_sensitivity_pair_results.csv",
    "summary_by_buffer": OUTPUT_DIR / "distance_sensitivity_summary_by_buffer.csv",
    "trend_stats": OUTPUT_DIR / "distance_sensitivity_trend_statistics.csv",
    "failed_nhda": OUTPUT_DIR / "failed_nhda.csv",
}

candidate_meta.to_csv(paths_to_save["candidate_metadata"], index=False)
df.to_csv(paths_to_save["pair_results"], index=False)
summary_by_buffer.to_csv(paths_to_save["summary_by_buffer"], index=False)
trend_df.to_csv(paths_to_save["trend_stats"], index=False)
pd.DataFrame({"nhda_id": sorted(set(failed_nhda))}).to_csv(paths_to_save["failed_nhda"], index=False)

for label, path in paths_to_save.items():
    print(f"✓ {label}: {path.name}")


# ============================================================================
# 7. PLOTS
# ============================================================================

print("\n7. CREATE PLOTS")
print("=" * 90)

fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharex=False)

sns.regplot(
    data=df,
    x="center_distance_m",
    y="abs_diff_lst_2025",
    scatter_kws={"alpha": 0.45, "s": 35},
    line_kws={"color": "darkred", "linewidth": 2},
    ax=axes[0],
)
axes[0].set_title("LST 2025: Absolute difference vs. NHDA-RA distance")
axes[0].set_xlabel("Distance between NHDA centroid and RA center [m]")
axes[0].set_ylabel("Absolute LST difference [°C]")

sns.regplot(
    data=df,
    x="center_distance_m",
    y="abs_diff_ndvi_2024",
    scatter_kws={"alpha": 0.45, "s": 35},
    line_kws={"color": "darkgreen", "linewidth": 2},
    ax=axes[1],
)
axes[1].set_title("NDVI 2024: Absolute difference vs. NHDA-RA distance")
axes[1].set_xlabel("Distance between NHDA centroid and RA center [m]")
axes[1].set_ylabel("Absolute NDVI difference")

plt.tight_layout()
plot_path_1 = OUTPUT_DIR / "distance_effect_scatter_regression.png"
plt.savefig(plot_path_1, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"✓ Plot: {plot_path_1.name}")

fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharex=True)

sns.lineplot(
    data=summary_by_buffer,
    x="buffer_m",
    y="mean_abs_diff_lst_2025",
    marker="o",
    linewidth=2.5,
    ax=axes[0],
)
axes[0].set_title("LST 2025: Mean absolute difference by distance band")
axes[0].set_xlabel("Reference-area distance band [m]")
axes[0].set_ylabel("Mean absolute LST difference [°C]")

sns.lineplot(
    data=summary_by_buffer,
    x="buffer_m",
    y="mean_abs_diff_ndvi_2024",
    marker="o",
    linewidth=2.5,
    ax=axes[1],
)
axes[1].set_title("NDVI 2024: Mean absolute difference by distance band")
axes[1].set_xlabel("Reference-area distance band [m]")
axes[1].set_ylabel("Mean absolute NDVI difference")

plt.tight_layout()
plot_path_2 = OUTPUT_DIR / "distance_effect_lineplots.png"
plt.savefig(plot_path_2, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"✓ Plot: {plot_path_2.name}")

fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharex=True)

sns.boxplot(
    data=df,
    x="buffer_m",
    y="abs_diff_lst_2025",
    ax=axes[0],
)
axes[0].set_title("LST 2025: Absolute difference by distance band")
axes[0].set_xlabel("Reference-area distance band [m]")
axes[0].set_ylabel("Absolute LST difference [°C]")
axes[0].tick_params(axis="x", rotation=45)

sns.boxplot(
    data=df,
    x="buffer_m",
    y="abs_diff_ndvi_2024",
    ax=axes[1],
)
axes[1].set_title("NDVI 2024: Absolute difference by distance band")
axes[1].set_xlabel("Reference-area distance band [m]")
axes[1].set_ylabel("Absolute NDVI difference")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plot_path_3 = OUTPUT_DIR / "distance_effect_boxplots.png"
plt.savefig(plot_path_3, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"✓ Plot: {plot_path_3.name}")


# ============================================================================
# 8. QUICK INTERPRETATION
# ============================================================================

print("\n8. QUICK INTERPRETATION")
print("=" * 90)

for variable_name, col in [("LST 2025", "mean_abs_diff_lst_2025"), ("NDVI 2024", "mean_abs_diff_ndvi_2024")]:
    start_val = summary_by_buffer.iloc[0][col]
    end_val = summary_by_buffer.iloc[-1][col]
    delta = end_val - start_val
    print(f"{variable_name}: {start_val:.4f} -> {end_val:.4f} (change with distance: {delta:+.4f})")

for _, row in trend_df.iterrows():
    print(
        f"{row['variable']}: Spearman rho = {row['spearman_rho_distance_vs_absdiff']:.4f}, "
        f"Pearson r = {row['pearson_r_distance_vs_absdiff']:.4f}"
    )

print("\nDone.")
print(f"End: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")